In [1]:
"""
This script loads the source parquet file ONCE, groups by article title,
and splits into 3 deterministic parts saved to disk.

WHY 3 PARTS?
Coreference resolution (FastCoref LingMess) is computationally expensive
and can exhaust RAM on large corpora. By pre-splitting into 3 parts, each
notebook can process one part at a time, running coreference on manageable
chunks.

Both kg_with_coref.ipynb and kg_without_coref.ipynb load these same 3 parts
to ensure perfectly consistent article extraction — the ONLY difference
between the two pipelines is whether coreference resolution is applied.
"""

import pandas as pd
import pickle
import os

In [2]:
# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
# HuggingFace DPR Wikipedia passages parquet
PARQUET_PATH = "hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00000-of-00157.parquet"

N_PARTS = 3
OUTPUT_DIR = "./prepared_splits"

# ------------------------------------------------------------------
# LOAD
# ------------------------------------------------------------------
print(f"Loading parquet: {PARQUET_PATH}")
df = pd.read_parquet(PARQUET_PATH)
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

required_cols = ["title", "text"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

Loading parquet: hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00000-of-00157.parquet
Total rows: 133,856
Columns: ['id', 'text', 'title', 'embeddings']


In [3]:
# ------------------------------------------------------------------
# GROUP BY ARTICLE TITLE (keep articles intact)
# ------------------------------------------------------------------
grouped = df.groupby("title")
article_groups = [group for _, group in grouped]
print(f"Unique articles: {len(article_groups):,}")

# ------------------------------------------------------------------
# DETERMINISTIC SPLIT INTO 3 PARTS
# ------------------------------------------------------------------
part_size = len(article_groups) // N_PARTS

os.makedirs(OUTPUT_DIR, exist_ok=True)

for i in range(N_PARTS):
    start_idx = i * part_size
    end_idx   = len(article_groups) if i == N_PARTS - 1 else (i + 1) * part_size

    part_df = pd.concat(article_groups[start_idx:end_idx], ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f"part_{i+1}_of_{N_PARTS}.pkl")

    with open(out_path, "wb") as f:
        pickle.dump(part_df, f)

    print(f"  Part {i+1}: {len(part_df):,} rows  ({end_idx - start_idx} articles) → {out_path}")

# ------------------------------------------------------------------
# SAVE METADATA
# ------------------------------------------------------------------
meta = {
    "source_parquet": PARQUET_PATH,
    "total_rows": len(df),
    "unique_articles": len(article_groups),
    "n_parts": N_PARTS,
    "split_indices": [
        {"part": i+1, "start": i*part_size, "end": (len(article_groups) if i==N_PARTS-1 else (i+1)*part_size)}
        for i in range(N_PARTS)
    ]
}

meta_path = os.path.join(OUTPUT_DIR, "split_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    import json
    json.dump(meta, f, indent=2)

print(f"\nMetadata saved: {meta_path}")
print("\nDone. Both notebooks should load these .pkl files to ensure identical inputs.")

Unique articles: 4,352
  Part 1: 43,008 rows  (1450 articles) → ./prepared_splits/part_1_of_3.pkl
  Part 2: 41,086 rows  (1450 articles) → ./prepared_splits/part_2_of_3.pkl
  Part 3: 49,762 rows  (1452 articles) → ./prepared_splits/part_3_of_3.pkl

Metadata saved: ./prepared_splits/split_metadata.json

Done. Both notebooks should load these .pkl files to ensure identical inputs.
